# Transformer from Scratch — German → English Machine Translation

## Project Overview

In this notebook we implement the **Transformer** architecture (Vaswani et al., *"Attention Is All You Need"*, 2017) **from scratch** using PyTorch regardless of built-in `nn.Transformer`, and train it on a German→English parallel corpus.

### Dataset
| Split | Sentences |
|-------|-----------|
| Train | 160 239 |
| Valid | 7 283 |
| Test | 6 750 |

Each `.de` / `.en` file contains one lowercased sentence per line; lines at the same index are translation pairs.

### Notebook Structure
1. **Setup & Imports**
2. **Data Loading & Exploration**
3. **Tokenization & Vocabulary**
4. **Dataset & DataLoader**
5. **Positional Encoding**
6. **Multi-Head Attention**
7. **Transformer Encoder & Decoder**
8. **Full Transformer Model**
9. **Training Loop**
10. **Evaluation & BLEU Score**

## 1. Setup & Imports

In [ ]:
import os
import math
import random
from collections import Counter
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import numpy as np
import matplotlib.pyplot as plt

# ---------- reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Data Loading & Exploration

We load the parallel `.de` / `.en` files. Each line index corresponds to a translation pair.
The text is already **lowercased** and **detokenized**; we split on whitespace for a simple word-level tokenisation.

In [ ]:
DATA_DIR = "."  # data files live in the project root

def read_file(path: str) -> List[str]:
    """Read a text file and return a list of stripped lines."""
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in f]

# ---------- load all splits ----------
train_de = read_file(os.path.join(DATA_DIR, "train.de"))
train_en = read_file(os.path.join(DATA_DIR, "train.en"))
valid_de = read_file(os.path.join(DATA_DIR, "valid.de"))
valid_en = read_file(os.path.join(DATA_DIR, "valid.en"))
test_de  = read_file(os.path.join(DATA_DIR, "test.de"))
test_en  = read_file(os.path.join(DATA_DIR, "test.en"))

print(f"Train : {len(train_de):>7,} pairs")
print(f"Valid : {len(valid_de):>7,} pairs")
print(f"Test  : {len(test_de):>7,} pairs")
print()
print("Example pair (train[0]):")
print(f"  DE: {train_de[0]}")
print(f"  EN: {train_en[0]}")

In [ ]:
# ---------- sentence-length distribution ----------
def token_lengths(sentences: List[str]) -> List[int]:
    return [len(s.split()) for s in sentences]

train_de_lens = token_lengths(train_de)
train_en_lens = token_lengths(train_en)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_de_lens, bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("Train DE – sentence lengths")
axes[0].set_xlabel("# tokens"); axes[0].set_ylabel("count")
axes[1].hist(train_en_lens, bins=60, color="coral", edgecolor="white")
axes[1].set_title("Train EN – sentence lengths")
axes[1].set_xlabel("# tokens"); axes[1].set_ylabel("count")
plt.tight_layout(); plt.show()

print(f"DE  — mean: {np.mean(train_de_lens):.1f}, median: {np.median(train_de_lens):.0f}, "
      f"max: {max(train_de_lens)}, 95th pctl: {np.percentile(train_de_lens, 95):.0f}")
print(f"EN  — mean: {np.mean(train_en_lens):.1f}, median: {np.median(train_en_lens):.0f}, "
      f"max: {max(train_en_lens)}, 95th pctl: {np.percentile(train_en_lens, 95):.0f}")

## 3. Tokenization & Vocabulary

We use a simple **whitespace tokenizer** and build vocabularies from the training set only.

Special tokens:
| Token | Index | Purpose |
|-------|-------|---------|
| `<pad>` | 0 | Padding shorter sequences in a batch |
| `<sos>` | 1 | Start-of-sequence (fed as first decoder input) |
| `<eos>` | 2 | End-of-sequence (signals the decoder to stop) |
| `<unk>` | 3 | Unknown / out-of-vocabulary token |

We cap vocabulary size with `MIN_FREQ` — tokens that appear fewer than `MIN_FREQ` times are mapped to `<unk>`.

In [ ]:
# ---------- special tokens ----------
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3
SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]

MIN_FREQ = 2  # minimum frequency to keep a token


class Vocabulary:
    """Word-level vocabulary built from a list of sentences."""

    def __init__(self, sentences: List[str], min_freq: int = MIN_FREQ):
        self.itos = list(SPECIALS)          # index  → token
        self.stoi = {t: i for i, t in enumerate(SPECIALS)}  # token → index

        # count all tokens
        counter = Counter()
        for sent in sentences:
            counter.update(sent.split())

        # add tokens that meet the frequency threshold
        for token, freq in counter.most_common():
            if freq < min_freq:
                continue
            if token not in self.stoi:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)

    def __len__(self):
        return len(self.itos)

    def encode(self, sentence: str) -> List[int]:
        """Tokenize & convert a sentence string to a list of indices (no <sos>/<eos>)."""
        return [self.stoi.get(t, UNK_IDX) for t in sentence.split()]

    def decode(self, indices, skip_special: bool = True) -> str:
        """Convert a list/tensor of indices back to a string."""
        tokens = []
        for idx in indices:
            idx = int(idx)
            if skip_special and idx in (PAD_IDX, SOS_IDX, EOS_IDX):
                continue
            tokens.append(self.itos[idx] if idx < len(self.itos) else "<unk>")
        return " ".join(tokens)


# ---------- build vocabularies (training data only!) ----------
vocab_de = Vocabulary(train_de, min_freq=MIN_FREQ)
vocab_en = Vocabulary(train_en, min_freq=MIN_FREQ)

print(f"DE vocabulary size: {len(vocab_de):,}")
print(f"EN vocabulary size: {len(vocab_en):,}")
print()
print("Encoding example:")
sample = train_de[0]
encoded = vocab_de.encode(sample)
print(f"  original : {sample}")
print(f"  encoded  : {encoded[:15]}…")
print(f"  decoded  : {vocab_de.decode(encoded)}")

## 4. Dataset & DataLoader

We wrap the parallel sentences in a PyTorch `Dataset` that:
1. Encodes each sentence to token indices using the vocabularies built above.
2. Prepends `<sos>` and appends `<eos>` to both source and target.
3. Returns tensors ready for batching.

A custom `collate_fn` **pads** variable-length sequences in each batch so that every tensor in a batch has the same length (filled with `<pad>` = 0).

In [ ]:
class TranslationDataset(Dataset):
    """Parallel-corpus dataset that returns encoded + wrapped (sos/eos) tensor pairs."""

    def __init__(self, src_sentences: List[str], tgt_sentences: List[str],
                 src_vocab: Vocabulary, tgt_vocab: Vocabulary):
        assert len(src_sentences) == len(tgt_sentences)
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_ids = [SOS_IDX] + self.src_vocab.encode(self.src_sentences[idx]) + [EOS_IDX]
        tgt_ids = [SOS_IDX] + self.tgt_vocab.encode(self.tgt_sentences[idx]) + [EOS_IDX]
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


def collate_fn(batch):
    """Pad source and target sequences independently so each batch is rectangular."""
    src_batch, tgt_batch = zip(*batch)
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


# ---------- hyperparameters (data) ----------
BATCH_SIZE = 128
MAX_SEQ_LEN = 128  # for positional-encoding buffer; actual lengths are dynamic

# ---------- create datasets ----------
train_dataset = TranslationDataset(train_de, train_en, vocab_de, vocab_en)
valid_dataset = TranslationDataset(valid_de, valid_en, vocab_de, vocab_en)
test_dataset  = TranslationDataset(test_de,  test_en,  vocab_de, vocab_en)

# ---------- create data loaders ----------
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test  batches: {len(test_loader)}")
print()

# do sanity check
src_sample, tgt_sample = next(iter(train_loader))
print(f"Source batch shape: {src_sample.shape}  (batch, seq_len)")
print(f"Target batch shape: {tgt_sample.shape}")

## 5. Masking Utilities

The Transformer needs two types of masks:

| Mask | Shape | Purpose |
|------|-------|---------|
| **Padding mask** | `(batch, 1, 1, seq_len)` | Prevents attention to `<pad>` positions |
| **Causal (look-ahead) mask** | `(1, 1, seq_len, seq_len)` | Prevents the decoder from attending to future tokens |

Both masks are combined for the decoder self-attention.

In [ ]:
def make_pad_mask(seq: torch.Tensor, pad_idx: int = PAD_IDX) -> torch.Tensor:
    """
    Create a padding mask.
    Args:
        seq: (batch, seq_len) — token indices
    Returns:
        (batch, 1, 1, seq_len) boolean mask — True where token == pad_idx
    """
    return (seq == pad_idx).unsqueeze(1).unsqueeze(2)


def make_causal_mask(size: int) -> torch.Tensor:
    """
    Create a causal (look-ahead) mask for the decoder.
    Returns:
        (1, 1, size, size) boolean mask — True for positions that should be masked
    """
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask.unsqueeze(0).unsqueeze(0)


# ---------- quick visualisation ----------
causal = make_causal_mask(8).squeeze()
plt.figure(figsize=(4, 4))
plt.imshow(causal, cmap="Greys", interpolation="none")
plt.title("Causal mask (white = masked)")
plt.xlabel("Key position"); plt.ylabel("Query position")
plt.show()

## 6. Positional Encoding

Since the Transformer has **no recurrence or convolution**, we inject positional information via sinusoidal positional encodings (Vaswani et al. §3.5):

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (not learned).
    Adds position information to token embeddings.
    """

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)                       # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float() # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )                                                          # (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd  indices
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)

        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            (batch, seq_len, d_model) with positional encoding added
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

## 7. Multi-Head Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Multi-head attention runs $h$ parallel attention heads and concatenates the results:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W^O$$

> **TODO** — Implement `MultiHeadAttention` below.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism.

    TODO: implement __init__ and forward.
    Skeleton provided — fill in the body.
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        # TODO: define linear projections for Q, K, V and the output
        raise NotImplementedError("Implement MultiHeadAttention.__init__")

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: (batch, q_len, d_model)
            key:   (batch, k_len, d_model)
            value: (batch, k_len, d_model)
            mask:  broadcastable boolean mask (True = ignore)
        Returns:
            (batch, q_len, d_model), attention_weights
        """
        # TODO: project, split heads, compute scaled dot-product attention, concat
        raise NotImplementedError("Implement MultiHeadAttention.forward")

## 8. Feed-Forward Network & Encoder / Decoder Layers

Each Transformer layer contains:
- Multi-Head Attention (self-attention, and cross-attention in the decoder)
- Position-wise Feed-Forward Network: $\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$
- Residual connections + Layer Normalisation around each sub-layer

> **TODO** — Implement `PositionwiseFeedForward`, `EncoderLayer`, and `DecoderLayer`.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """Two-layer feed-forward network with ReLU activation."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: define two linear layers and dropout
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


class EncoderLayer(nn.Module):
    """Single Transformer encoder layer: self-attn → add & norm → FFN → add & norm."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: MultiHeadAttention, PositionwiseFeedForward, LayerNorm, Dropout
        raise NotImplementedError

    def forward(self, src, src_mask):
        # TODO
        raise NotImplementedError


class DecoderLayer(nn.Module):
    """
    Single Transformer decoder layer:
        masked-self-attn → add & norm →
        cross-attn (over encoder output) → add & norm →
        FFN → add & norm
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, tgt, enc_out, tgt_mask, src_mask):
        # TODO
        raise NotImplementedError

## 9. Full Transformer Model

Assemble the encoder stack and decoder stack into a single `Transformer` module.

> **TODO** — Implement the `Transformer` class below.

In [ ]:
# ---------- model hyperparameters ----------
D_MODEL   = 256    # embedding / hidden dimension
N_HEADS   = 8      # number of attention heads
N_LAYERS  = 3      # number of encoder (and decoder) layers
D_FF      = 512    # feed-forward inner dimension
DROPOUT   = 0.1
SRC_VOCAB = len(vocab_de)
TGT_VOCAB = len(vocab_en)


class Transformer(nn.Module):
    """
    Full encoder-decoder Transformer for sequence-to-sequence translation.

    Components:
        - Source & target token embeddings (scaled by √d_model)
        - Positional encoding (shared or separate)
        - N encoder layers
        - N decoder layers
        - Final linear projection to target vocabulary
    """

    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_heads,
                 n_layers, d_ff, dropout, pad_idx):
        super().__init__()
        # TODO: define embeddings, positional encoding, encoder layers,
        #       decoder layers, output projection
        raise NotImplementedError("Implement Transformer.__init__")

    def forward(self, src, tgt):
        """
        Args:
            src: (batch, src_len) — source token indices
            tgt: (batch, tgt_len) — target token indices (teacher-forced)
        Returns:
            (batch, tgt_len, tgt_vocab_size) — logits over target vocabulary
        """
        # TODO: build masks, encode, decode, project
        raise NotImplementedError("Implement Transformer.forward")


# Uncomment once the model is implemented:
# model = Transformer(SRC_VOCAB, TGT_VOCAB, D_MODEL, N_HEADS,
#                     N_LAYERS, D_FF, DROPOUT, PAD_IDX).to(DEVICE)
# print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")